In [1]:
import torch
from sklearn.datasets import fetch_california_housing
from src.mssp import MSSP

In [2]:
RANDOM_SEED = 42
if RANDOM_SEED is not None:
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed(RANDOM_SEED)

In [3]:
data = fetch_california_housing()
X = torch.tensor(data.data, dtype=torch.double)
y = torch.tensor(data.target, dtype=torch.double)

i = torch.randperm(len(X))
i_train = i[:int(len(X)*0.8)]
i_valid = i[int(len(X)*0.8):int(len(X)*0.9)]
i_test = i[int(len(X)*0.9):]

X_train, y_train = X[i_train], y[i_train]
X_valid, y_valid = X[i_valid], y[i_valid]
X_test, y_test = X[i_test], y[i_test]

In [ ]:
models = []
y_pred = torch.zeros_like(y_train)
patience = 5
best_err = float("inf")

for i in range(500):
    residual = y_train - y_pred
    shift = -residual.min() + 1e-8 
    residual += shift

    mssp = MSSP(
    n_best=100, 
    loss_fn="mse", 
    random_seed=RANDOM_SEED, 
    epochs=5, 
    diversity_ratio=0.75, 
    pow_cross=True,
    )

    mssp.fit(X_train, residual)
    pred = mssp.predict(X_train)
    pred -= shift
    y_pred += 0.1 * pred
    models.append(mssp)

    err = (y_pred - y_train).abs().mean()
    print(f"ERR {i}: {err}")

    if err < best_err - 1e-6:
        best_err = err
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at iter {i}")
        break

loss (mse): 0.6163414716720581 epoch: 0 , time: 0.61s
loss (mse): 0.46785733103752136 epoch: 1 , time: 1.34s
loss (mse): 0.45522540807724 epoch: 2 , time: 1.29s
loss (mse): 0.4490717351436615 epoch: 3 , time: 1.27s
loss (mse): 0.4415287375450134 epoch: 4 , time: 1.22s
Best loss: 0.4415287375450134 after training for 4 epochs
ERR 0: 0.890204967424818
loss (mse): 0.5804042220115662 epoch: 0 , time: 0.55s
loss (mse): 0.46160319447517395 epoch: 1 , time: 1.28s
loss (mse): 0.4509318470954895 epoch: 2 , time: 1.31s
loss (mse): 0.4433182179927826 epoch: 3 , time: 1.40s
loss (mse): 0.44020336866378784 epoch: 4 , time: 1.48s
Best loss: 0.44020336866378784 after training for 4 epochs
ERR 1: 0.7920181485931614
loss (mse): 0.5513337254524231 epoch: 0 , time: 0.51s
loss (mse): 0.4642719626426697 epoch: 1 , time: 1.27s
loss (mse): 0.4564211368560791 epoch: 2 , time: 1.29s
loss (mse): 0.44757574796676636 epoch: 3 , time: 1.20s
loss (mse): 0.442300409078598 epoch: 4 , time: 1.34s
Best loss: 0.44230040

In [ ]:
print("ERR", (y_pred - y_train).abs().mean())

ERR tensor(0.4660, dtype=torch.float64)


## След обучение, mssp вече има history обект, съхраняващ нужните данни за изграждане на граф

In [6]:
mssp.history

[{'type': 'primitives',
  'epoch': -1,
  'params': [(tensor([0.4738]), tensor(3.5825), 'lin', 0),
   (tensor([0.0633]), tensor(3.7948), 'lgn', 0),
   (tensor([0.0697]), tensor(1.2741), 'xpy', 0),
   (tensor([0.0120]), tensor(1.3098), 'pow', 0),
   (tensor([-1.1284e-07]), tensor(3.6923), 'rex', 0),
   (tensor([214351.0156]), tensor(-43580.3398), 'rey', 0),
   (tensor([0.1137]), tensor(1.8880), 'sqr', 0),
   (tensor([0.5219]), tensor(3.5737), 'snx', 0),
   (tensor([0.0676]), tensor(3.6556), 'lin', 1),
   (tensor([0.0160]), tensor(3.7044), 'lgn', 1),
   (tensor([0.0084]), tensor(1.2857), 'xpy', 1),
   (tensor([0.0017]), tensor(1.2916), 'pow', 1),
   (tensor([1.9397e-07]), tensor(3.6922), 'rex', 1),
   (tensor([742.9650]), tensor(5654.0767), 'rey', 1),
   (tensor([0.0131]), tensor(1.9073), 'sqr', 1),
   (tensor([0.0718]), tensor(3.6563), 'snx', 1),
   (tensor([1.2784]), tensor(3.6508), 'lin', 2),
   (tensor([0.0467]), tensor(3.8552), 'lgn', 2),
   (tensor([0.2236]), tensor(1.2830), 'xpy', 

## За изграждане на граф се използва методът _buiuld_model(top_k). Топ к позволява изграждането на повече от един граф. Топ к = 2 означава да се изградят 2те най-добри решения 

In [ ]:
mssp._build_model(4)

## В масивът model на mssp обекта се съхраняват графите, като са сортирани (на първо място е най-добрия модел)

In [ ]:
[type(m) for m in mssp.model]

## Всеки граф има пропърти head от тип Node. Това е входния Node. Всеки Node има left_child и right_child, които са или от тип Node или от тип PrimitiveNode (означава, че е достигнато последното ниво)

In [ ]:
graph = mssp.model[0]

In [ ]:
type(graph.head)

In [ ]:
type(graph.head.left_child)

In [ ]:
type(graph.head.right_child)

In [ ]:
node = graph.head
for i in range(4):
    node = node.left_child
    print(type(node))

## Има възможност за визуализиране на всеки граф. top_k определя ранкът на графа.

In [ ]:
mssp.plot_graph(top_k=1)

In [ ]:
mssp.plot_graph(top_k=2)